# **2일차 과제 — 기본 Agent 구현하기**

> **제출자: 조태희** · OZ코딩스쿨 AI 헬스케어 부트캠프 6기 · 자연어 처리(NLP) 2일차 과제 (AI Agent 기초1)

**📢 강사님 공지 (과제 페이지)** — "문제 2, 4번 제외 (3일차에 학습하고 진행하는 걸로 하겠습니다) 문제 1, 3번만 하시고 제출 해주시면 되겠습니다!"
→ 이번 제출의 **필수는 문제 1, 3번**입니다. 문제 2, 4번 풀이도 복습용으로 함께 넣어 두었으니(제목에 **선택** 표시) 3일차 수업 후에 확인해 보세요.

**📌 실행 안내**
- 실행 번호와 출력이 있는 셀은 API 키 없이도 돌아가는 셀(State 정의, 노드 함수 정의, 그래프 조립·구조 확인)이라 미리 실행해 둔 결과입니다.
- 코드 첫 줄에 **🔑 [Colab에서 실행]** 표시가 있는 셀은 OpenAI API 키가 필요해서 아직 실행하지 않았습니다. Colab에서 `api_key.txt`를 업로드하거나(또는 왼쪽 🔑 보안 비밀에 `OPENAI_API_KEY` 등록) **런타임 → 모두 실행**으로 위에서부터 차례대로 실행해 주세요.
- GPT 답변은 실행할 때마다 표현이 조금씩 달라질 수 있습니다.

# **1. 환경준비**

## **(1) 라이브러리**

In [ ]:
# 🔑 [Colab에서 실행] 필요한 라이브러리 설치 (-q: 설치 로그를 짧게 표시)
!pip install langchain-openai langgraph -q

In [1]:
import os                                 # 환경변수(API 키) 관리
import ast                                # 문자열 → 파이썬 자료형(딕셔너리) 변환
import operator                           # operator.add(리스트 이어붙이기)를 '쌓기' 규칙으로 쓰기 위해
from typing import TypedDict, Annotated   # State의 키·타입을 정의하는 도구 (Annotated: 타입에 '추가 규칙'을 붙임)

from langchain_openai import ChatOpenAI                          # OpenAI GPT 모델
from langchain_core.messages import HumanMessage, SystemMessage  # 메시지 종류 (사용자 요청 / 역할·규칙 지침)
from langgraph.graph import StateGraph, START, END               # 그래프 설계도, 시작점, 끝점

## **(2) OpenAI API Key 확인**

In [ ]:
# 🔑 [Colab에서 실행] api_key.txt 파일 업로드
#   - api_key.txt 안에는 OPENAI_API_KEY=sk-... 형식으로 적혀 있어야 합니다.
#   - Colab 왼쪽 🔑(보안 비밀)에 OPENAI_API_KEY를 등록해 두었다면 이 셀은 건너뛰어도 됩니다.
from google.colab import files   # Colab에서 내 컴퓨터 파일을 올리는 기능
uploaded = files.upload()        # 업로드 창이 뜨면 api_key.txt를 선택합니다

In [ ]:
# 🔑 [Colab에서 실행] API 키를 환경변수로 등록합니다 (코드에 키를 직접 적지 않기 위해서예요)
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:                      # 키 파일을 읽기 모드로 엽니다
        for line in f:                                  # 한 줄씩 읽으면서
            line = line.strip()                         # 앞뒤 공백·줄바꿈을 지우고
            if line and "=" in line:                    # 빈 줄이 아니고 "이름=값" 형태라면
                key, value = line.split("=", 1)         # 첫 번째 '='를 기준으로 이름과 값을 나눠서
                os.environ[key.strip()] = value.strip() # 환경변수로 등록합니다

def ensure_api_key(name):
    """환경변수에 키가 없으면 Colab 보안 비밀 → 직접 입력 순서로 채웁니다."""
    if os.environ.get(name):                            # 이미 등록돼 있으면 그대로 사용
        return                                          # 더 할 일이 없으니 함수 종료
    try:
        from google.colab import userdata               # Colab 🔑 보안 비밀 저장소
        os.environ[name] = userdata.get(name)           # 보안 비밀에 등록된 키를 꺼내 등록
    except Exception:                                   # 보안 비밀에 없거나 Colab이 아니면
        from getpass import getpass                     # 입력한 글자가 화면에 안 보이는 입력창
        os.environ[name] = getpass(f"{name}를 입력하세요: ")   # 직접 입력받아 등록


if os.path.exists("api_key.txt"):                       # 업로드한 키 파일이 있으면
    load_api_keys("api_key.txt")                        # 파일에서 읽어서 등록
ensure_api_key("OPENAI_API_KEY")                        # 그래도 없으면 보안 비밀/직접 입력으로 채움

* 실행해서 key가 제대로 보이는지 확인하고 삭제하세요.

In [ ]:
# 🔑 [Colab에서 실행] 키가 잘 들어왔는지 앞부분만 확인합니다
#   (전체 키가 노트북에 남으면 유출될 수 있어서 앞 7글자만 보이게 했습니다. 확인 후 이 셀은 삭제하세요.)
print(os.environ['OPENAI_API_KEY'][:7] + "...")

In [ ]:
# 🔑 [Colab에서 실행] GPT 모델 준비 (API 키가 있어야 객체를 만들 수 있습니다)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)   # temperature=0 → 무작위성을 최소로 해서 답이 흔들리지 않게 함

# **2. 과제**

## 문제 1. State 정의 + 단일 노드 그래프 — 환자 기본정보 저장

* 접수 메모(자유 텍스트)를 입력받아 환자 이름/나이/주 증상을 추출해서 State에 저장하는 단일 노드 그래프를 구현하세요.
* 입력 예시
    * `"환자 이름은 김민수이고 나이는 58세입니다. 최근 계단을 오르면 가슴이 답답하고 숨이 찬 증상을 호소하고 있습니다."`
* 요구사항
    * State: `messages`(쌓기), `patient_name`(덮어쓰기), `age`(덮어쓰기), `chief_complaint`(덮어쓰기)
    * 단일 노드에서 LLM에게 딕셔너리 형식으로만 답하도록 지시하고, `ast.literal_eval`로 파싱해서 State에 반영
    * `graph.invoke()`로 1회 실행

**💡 풀이 포인트** — LangGraph의 **State(상태)**는 모든 노드가 함께 보는 **공유 메모장**입니다. `messages`처럼 대화가 계속 쌓여야 하는 칸에는 `Annotated[list, operator.add]`로 "기존 리스트 + 새 리스트" 규칙(**리듀서**, reducer: 새 값을 기존 값과 어떻게 합칠지 정한 규칙)을 붙이고, 이름·나이처럼 최신 값 하나면 되는 칸은 타입만 적어서 **덮어쓰기** 되게 합니다. **노드**(node, 그래프 안의 작업 단위 함수)는 State를 받아서 **바꾸고 싶은 칸만 딕셔너리로 반환**하면, LangGraph가 그 규칙대로 메모장에 반영해 줍니다.

In [2]:
# [문제 1] State 정의 — 그래프 전체가 함께 쓰는 '공유 메모장'의 칸(키)을 정합니다
class PatientState(TypedDict):
    messages: Annotated[list, operator.add]   # 대화 기록 → operator.add 리듀서로 '쌓기' (기존 리스트 + 새 리스트)
    patient_name: str                          # 환자 이름 → 리듀서 없음 = 새 값으로 '덮어쓰기'
    age: int                                   # 나이 → 덮어쓰기
    chief_complaint: str                       # 주 증상(chief complaint, 환자가 호소하는 가장 주된 증상) → 덮어쓰기

In [3]:
def parse_dict(text):
    """LLM 답변 문자열에서 { } 부분만 잘라 ast.literal_eval로 딕셔너리로 바꿉니다."""
    start, end = text.find("{"), text.rfind("}")    # 중괄호 시작·끝 위치 (```로 감싸서 답해도 안전하게)
    return ast.literal_eval(text[start:end + 1])    # 문자열 → 딕셔너리


def extract_patient_info(state: PatientState):
    # 시스템 메시지: 무엇을, 어떤 형식(딕셔너리)으로 뽑을지 지시합니다
    s_msg = SystemMessage(content="""당신은 병원 접수 메모에서 환자 기본정보를 추출하는 AI입니다.
접수 메모를 읽고 아래 형식의 파이썬 딕셔너리로만 답하세요. 다른 설명이나 코드블록(```)은 붙이지 마세요.
{'patient_name': '환자 이름', 'age': 나이(정수 숫자만), 'chief_complaint': '주 증상을 한 문장으로 요약'}""")

    response = llm.invoke([s_msg] + state["messages"])   # 지침 + State에 쌓인 메시지(접수 메모)를 함께 전달
    info = parse_dict(response.content)                   # 문자열 답변 → 딕셔너리 (ast.literal_eval)

    # 바꾸고 싶은 칸만 반환하면 LangGraph가 State에 반영합니다
    return {
        "messages": [response],                        # 쌓기: AI 응답을 대화 기록 뒤에 추가 (리스트로 감싸야 이어붙여짐)
        "patient_name": info["patient_name"],          # 덮어쓰기
        "age": int(str(info["age"]).rstrip("세")),     # 덮어쓰기 (혹시 '58세'처럼 와도 정수로 변환)
        "chief_complaint": info["chief_complaint"],    # 덮어쓰기
    }

In [4]:
builder = StateGraph(PatientState)                              # PatientState를 메모장으로 쓰는 그래프 설계도 만들기
builder.add_node("extract_patient_info", extract_patient_info)  # 노드 등록: ("노드 이름", 실행할 함수)
builder.add_edge(START, "extract_patient_info")                 # 시작 → 정보 추출 노드
builder.add_edge("extract_patient_info", END)                   # 정보 추출 노드 → 끝
graph = builder.compile()                                       # 설계도를 실행 가능한 그래프로 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

__start__ → extract_patient_info 
extract_patient_info → __end__ 


In [ ]:
# 🔑 [Colab에서 실행] 그래프 1회 실행 (노드 안에서 LLM을 호출하므로 API 키 필요)
# 문제에서 준 접수 메모 예시
memo = "환자 이름은 김민수이고 나이는 58세입니다. 최근 계단을 오르면 가슴이 답답하고 숨이 찬 증상을 호소하고 있습니다."

result = graph.invoke({"messages": [HumanMessage(content=memo)]})   # 처음 State: messages에 접수 메모만 넣고 실행

print("환자 이름:", result["patient_name"])
print("나이:", result["age"], type(result["age"]))   # 정수(int)로 저장됐는지 타입까지 확인
print("주 증상:", result["chief_complaint"])
print("쌓인 메시지 수:", len(result["messages"]))     # 접수 메모(Human) 1개 + AI 응답 1개 = 2개

## 문제 2. State에 메시지 누적하며 멀티턴 대화 실행

* `messages` 키만 갖는 State로 챗봇 노드를 구현하고, 이전 대화 내용을 이어서 2턴 실행하세요.
* 입력 예시
    * 1턴: `"계단을 오르면 숨이 차요. 왜 그런 걸까요?"`
    * 2턴: `"그럼 지금 병원에 가야 할까요?"`
* 요구사항
    * State: `messages`(쌓기)만 사용
    * 1턴 실행 결과의 `messages`를 2턴 입력에 그대로 이어붙여서, 2턴째 실행 시 이전 대화가 함께 전달되도록 구성
    * 최종 `messages`를 순서대로 출력해서 누적 여부 확인

> **선택 문제** — 강사님 공지에 따라 3일차 학습 후 진행하는 문제입니다(이번 제출 필수 아님). 복습용으로 풀이를 넣어 두었습니다.

**💡 풀이 포인트** — 그래프는 `invoke`가 한 번 끝나면 State를 기억하지 않습니다. 그래서 1턴 결과의 `messages`(질문+답변)를 꺼내 2턴 질문을 뒤에 붙인 다음 다시 넣어 주면, GPT가 이전 대화까지 함께 보고 답할 수 있습니다. (3일차에서는 이 과정을 자동으로 해주는 `MemorySaver`를 배웁니다.)

In [5]:
# [문제 2] messages 칸 하나만 있는 State
class ChatState(TypedDict):
    messages: Annotated[list, operator.add]   # 대화 기록 '쌓기'

In [6]:
def chatbot(state: ChatState):
    # 지침: 이전 대화를 참고해서 답하는 건강 상담 챗봇
    s_msg = SystemMessage(content="당신은 환자의 건강 상담을 돕는 친절한 의료 상담 챗봇입니다. "
                                  "이전 대화 내용을 참고해서 답하고, 진단을 단정하지 말고 필요하면 진료를 권하세요.")
    response = llm.invoke([s_msg] + state["messages"])   # 지침 + 지금까지 쌓인 모든 대화를 함께 전달
    return {"messages": [response]}                       # AI 답변을 대화 기록에 '쌓기'

In [7]:
builder = StateGraph(ChatState)              # 새 그래프 설계도
builder.add_node("chatbot", chatbot)         # 챗봇 노드 등록
builder.add_edge(START, "chatbot")           # 시작 → 챗봇
builder.add_edge("chatbot", END)             # 챗봇 → 끝
chat_graph = builder.compile()               # 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = chat_graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

__start__ → chatbot 
chatbot → __end__ 


In [ ]:
# 🔑 [Colab에서 실행] 2턴 대화 실행 (API 키 필요)
# 1턴: 첫 질문만 넣고 실행
turn1 = chat_graph.invoke({"messages": [HumanMessage(content="계단을 오르면 숨이 차요. 왜 그런 걸까요?")]})

# 2턴: 1턴 결과 messages(질문+답변) 뒤에 새 질문을 이어붙여서 실행 → 이전 대화가 함께 전달됨
turn2_input = turn1["messages"] + [HumanMessage(content="그럼 지금 병원에 가야 할까요?")]
turn2 = chat_graph.invoke({"messages": turn2_input})

# 최종 messages를 순서대로 출력해서 누적됐는지 확인
for i, m in enumerate(turn2["messages"], start=1):   # 1번부터 번호를 붙여서
    print(f"[{i}] {m.type}: {m.content}\n")           # 메시지 종류(human/ai)와 내용
print("총 메시지 수:", len(turn2["messages"]))        # 1턴(질문·답변) 2개 + 2턴(질문·답변) 2개 = 4개

## 문제 3. 여러 키를 갖는 State 설계·업데이트 — 복약 순응도 상담 파이프라인

* 환자의 복약 습관 설명을 입력받아, 2개 노드로 문제점과 개선방안을 순서대로 도출하세요.
* 입력 예시
    * `"요즘 아침 약 먹는 걸 자꾸 깜빡해요. 저녁 약은 그나마 챙기는 편인데, 주말엔 아예 안 먹을 때도 있어요."`
* 처리 흐름
    * `issue_agent` : 복약 순응도 문제점 1가지 도출 → `issue`
    * `suggestion_agent` : `issue`를 참고해서 개선 방안 1가지 제안 → `suggestion`
* 요구사항
    * State: `messages`(쌓기), `user_input`(덮어쓰기), `issue`(덮어쓰기), `suggestion`(덮어쓰기)
    * 2개 노드를 순서대로 연결, `graph.invoke()`로 1회 실행, 최종 State 출력

**💡 풀이 포인트** — 노드끼리는 **State를 통해서만** 정보를 주고받습니다. `issue_agent`가 State의 `issue` 칸에 문제점을 적어 두면, 다음 노드 `suggestion_agent`가 그 칸을 읽어서 개선 방안을 만드는 구조예요(릴레이 경주에서 바통을 넘기는 것과 같아요). 그래서 두 노드를 **엣지**(edge, 노드 사이의 연결선)로 `START → issue_agent → suggestion_agent → END` 순서로 연결합니다. 복약 순응도(adherence)는 "처방받은 대로 약을 잘 챙겨 먹는 정도"를 뜻합니다.

In [8]:
# [문제 3] 여러 칸을 가진 State
class AdherenceState(TypedDict):
    messages: Annotated[list, operator.add]   # 각 노드가 주고받은 메시지 기록 → 쌓기
    user_input: str                            # 환자의 복약 습관 설명 → 덮어쓰기
    issue: str                                 # issue_agent가 찾은 문제점 → 덮어쓰기
    suggestion: str                            # suggestion_agent가 낸 개선 방안 → 덮어쓰기

In [9]:
def issue_agent(state: AdherenceState):
    # 역할: 복약 습관 설명에서 '문제점 1가지'만 찾기
    s_msg = SystemMessage(content="당신은 복약 상담 전문 약사입니다. 환자의 복약 습관 설명을 읽고, "
                                  "복약 순응도(처방대로 약을 잘 먹는 정도)에서 가장 중요한 문제점 1가지만 한 문장으로 답하세요.")
    h_msg = HumanMessage(content=state["user_input"])     # State에서 환자 설명을 꺼내 요청서로 만듦
    response = llm.invoke([s_msg, h_msg])                 # GPT 호출
    return {"messages": [h_msg, response],                # 보낸 요청과 받은 답을 기록에 쌓기
            "issue": response.content}                    # 문제점을 issue 칸에 저장 (다음 노드가 읽음)


def suggestion_agent(state: AdherenceState):
    # 역할: 앞 노드가 찾은 issue를 참고해 '개선 방안 1가지' 제안
    s_msg = SystemMessage(content="당신은 복약 상담 전문 약사입니다. 주어진 복약 문제점을 해결할 수 있는, "
                                  "환자가 바로 실천 가능한 개선 방안 1가지를 1~2문장으로 제안하세요.")
    h_msg = HumanMessage(content=f"[환자 설명]\n{state['user_input']}\n\n[문제점]\n{state['issue']}")   # 앞 노드 결과 활용
    response = llm.invoke([s_msg, h_msg])                 # GPT 호출
    return {"messages": [h_msg, response],                # 기록 쌓기
            "suggestion": response.content}               # 개선 방안을 suggestion 칸에 저장

In [10]:
builder = StateGraph(AdherenceState)                  # 새 그래프 설계도
builder.add_node("issue_agent", issue_agent)          # 노드 1: 문제점 도출
builder.add_node("suggestion_agent", suggestion_agent)  # 노드 2: 개선 방안 제안
builder.add_edge(START, "issue_agent")                # 시작 → 문제점 도출
builder.add_edge("issue_agent", "suggestion_agent")   # 문제점 도출 → 개선 방안 (순서대로 연결)
builder.add_edge("suggestion_agent", END)             # 개선 방안 → 끝
adherence_graph = builder.compile()                   # 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = adherence_graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

__start__ → issue_agent 
issue_agent → suggestion_agent 
suggestion_agent → __end__ 


In [ ]:
# 🔑 [Colab에서 실행] 파이프라인 1회 실행 후 최종 State 출력 (API 키 필요)
# 문제에서 준 복약 습관 설명 예시
user_input = "요즘 아침 약 먹는 걸 자꾸 깜빡해요. 저녁 약은 그나마 챙기는 편인데, 주말엔 아예 안 먹을 때도 있어요."

result3 = adherence_graph.invoke({"user_input": user_input, "messages": []})   # 처음 State: 환자 설명 + 빈 기록

# 최종 State의 각 칸 확인
print("[user_input]", result3["user_input"])
print("[issue]", result3["issue"])
print("[suggestion]", result3["suggestion"])
print("[messages 개수]", len(result3["messages"]))   # 노드마다 (요청 1 + 답변 1) × 2개 노드 = 4개

## 문제 4. 조건에 따라 다른 응답 분기 처리 — 증상 긴급도 라우팅

* 증상 텍스트의 긴급도를 판단해서, 응급/일반에 따라 서로 다른 노드로 분기하세요.
* 입력 예시
    * 응급: `"갑자기 가슴을 짓누르는 듯한 통증과 함께 숨쉬기가 힘들어요."`
    * 일반: `"콧물이 나고 목이 약간 칼칼해요."`
* 처리 흐름
    * `triage_router` : 증상을 `"응급"` 또는 `"일반"` 중 하나로 판단 → `urgency`
    * `urgency`가 `"응급"`이면 `emergency_response` 노드로, `"일반"`이면 `general_response` 노드로 분기
* 요구사항
    * State: `messages`(쌓기), `symptom_text`(덮어쓰기), `urgency`(덮어쓰기), `response`(덮어쓰기)
    * SystemMessage에 `"응급"`/`"일반"` 둘 중 하나만 출력하도록 지침 명시
    * 조건 분기 함수 1개 작성 후 `add_conditional_edges`로 연결, 두 입력 각각 실행해서 분기 확인

> **선택 문제** — 강사님 공지에 따라 3일차 학습 후 진행하는 문제입니다(이번 제출 필수 아님). 복습용으로 풀이를 넣어 두었습니다.

**💡 풀이 포인트** — `add_conditional_edges`는 갈림길 표지판입니다. 노드가 끝나면 **분기 함수**가 State(`urgency`)를 보고 "길 이름"(`"emergency"`/`"general"`)을 돌려주고, 그 이름에 연결된 노드로 이동합니다. GPT가 가끔 "응급입니다."처럼 군더더기를 붙일 수 있어서, 답에 "응급"이 들어 있는지로 두 값 중 하나로 정리한 뒤 State에 저장했습니다. 분기 함수는 LLM 없이도 테스트할 수 있어서 직접 만든 State로 미리 확인해 두었습니다.

In [11]:
# [문제 4] 긴급도 분기용 State
class TriageState(TypedDict):
    messages: Annotated[list, operator.add]   # 메시지 기록 → 쌓기
    symptom_text: str                          # 환자 증상 설명 → 덮어쓰기
    urgency: str                               # "응급" 또는 "일반" → 덮어쓰기
    response: str                              # 최종 안내 문구 → 덮어쓰기

In [12]:
def triage_router(state: TriageState):
    # 지침에 '응급/일반 둘 중 하나만 출력'을 분명히 적습니다
    s_msg = SystemMessage(content="""당신은 응급실 분류(트리아지) 간호사입니다. 환자의 증상 설명을 읽고 긴급도를 판단하세요.
- 응급: 가슴 통증, 호흡곤란, 의식 저하, 심한 출혈처럼 즉시 진료가 필요한 경우
- 일반: 그 외 당장 생명에 위협이 되지 않는 가벼운 증상
반드시 "응급" 또는 "일반" 두 단어 중 하나만 출력하세요. 다른 말이나 문장부호는 절대 붙이지 마세요.""")
    response = llm.invoke([s_msg, HumanMessage(content=state["symptom_text"])])   # GPT에게 긴급도 판단 요청
    urgency = "응급" if "응급" in response.content else "일반"                      # 군더더기가 붙어도 두 값 중 하나로 정리
    return {"messages": [response], "urgency": urgency}                             # 판단 결과를 urgency 칸에 저장


def emergency_response(state: TriageState):
    # 응급일 때: 즉시 119/응급실 안내
    s_msg = SystemMessage(content="당신은 응급 상황 안내 담당자입니다. 환자가 즉시 119에 신고하거나 "
                                  "가까운 응급실을 방문하도록 2~3문장으로 침착하고 단호하게 안내하세요.")
    response = llm.invoke([s_msg, HumanMessage(content=state["symptom_text"])])   # 증상을 보내고 안내 문구 받기
    return {"messages": [response], "response": response.content}


def general_response(state: TriageState):
    # 일반일 때: 자가 관리 + 악화 시 병원 방문 안내
    s_msg = SystemMessage(content="당신은 친절한 외래 상담 간호사입니다. 집에서 할 수 있는 관리 방법과 "
                                  "증상이 심해지면 병원을 방문하라는 안내를 2~3문장으로 하세요.")
    response = llm.invoke([s_msg, HumanMessage(content=state["symptom_text"])])   # 증상을 보내고 안내 문구 받기
    return {"messages": [response], "response": response.content}

In [13]:
def route_by_urgency(state: TriageState) -> str:
    # 분기 함수: urgency 값을 보고 다음에 갈 '길 이름'을 돌려줍니다
    if state["urgency"] == "응급":
        return "emergency"
    return "general"


builder = StateGraph(TriageState)                             # 새 그래프 설계도
builder.add_node("triage_router", triage_router)              # 긴급도 판단 노드
builder.add_node("emergency_response", emergency_response)    # 응급 안내 노드
builder.add_node("general_response", general_response)        # 일반 안내 노드
builder.add_edge(START, "triage_router")                      # 시작 → 긴급도 판단
builder.add_conditional_edges(
    "triage_router",                                          # 이 노드가 끝나면
    route_by_urgency,                                         # 이 함수로 길을 고르고
    {"emergency": "emergency_response",                       # "emergency" → 응급 안내 노드
     "general": "general_response"},                          # "general"   → 일반 안내 노드
)
builder.add_edge("emergency_response", END)                   # 응급 안내 → 끝
builder.add_edge("general_response", END)                     # 일반 안내 → 끝
triage_graph = builder.compile()                              # 완성

# 그래프 연결 구조 확인 (__start__ = START, __end__ = END)
drawn = triage_graph.get_graph()                                              # 그래프 구조(노드·엣지) 정보 꺼내기
node_order = list(drawn.nodes)                                       # 노드를 등록한 순서
for edge in sorted(drawn.edges, key=lambda e: node_order.index(e.source)):   # 출발 노드 순서대로 정렬해서 하나씩
    label = ""                                                       # 기본값: 조건 표시 없음
    if edge.conditional:                                             # 조건부 연결이면 어떤 조건인지 표시
        label = f"   (조건부: {edge.data})" if edge.data else "   (조건부)"
    print(edge.source, "→", edge.target, label)

# 분기 함수만 따로 테스트 (LLM 없이, 직접 만든 State로 어느 길로 가는지 확인)
print("\nurgency='응급' →", route_by_urgency({"urgency": "응급"}))
print("urgency='일반' →", route_by_urgency({"urgency": "일반"}))

__start__ → triage_router 
triage_router → emergency_response    (조건부: emergency)
triage_router → general_response    (조건부: general)
emergency_response → __end__ 
general_response → __end__ 

urgency='응급' → emergency
urgency='일반' → general


In [ ]:
# 🔑 [Colab에서 실행] 두 입력을 각각 실행해서 분기 확인 (API 키 필요)
test_symptoms = [
    "갑자기 가슴을 짓누르는 듯한 통증과 함께 숨쉬기가 힘들어요.",   # 응급 예시
    "콧물이 나고 목이 약간 칼칼해요.",                               # 일반 예시
]

for symptom in test_symptoms:                                   # 입력 2개를 차례로
    result4 = triage_graph.invoke({"symptom_text": symptom, "messages": []})   # 증상마다 그래프 실행
    print("증상:", symptom)
    print("판단(urgency):", result4["urgency"])                  # 응급 / 일반
    print("안내(response):", result4["response"])                # 분기된 노드가 만든 안내 문구
    print("-" * 60)